# База на телефоне, индексатор на ПК

Как работать с базой, запущенной на телефоне (Termux), с другого устройства — например, индексировать её Claude Code с ПК. Три способа пробросить порт 8888, что выбрать и что учесть.

## В чём задача

Индексатор на ПК — обычный MCP-клиент; ему нужен доступ к порту 8888 телефона. По умолчанию Jupyter слушает только `127.0.0.1` (`jupyter_server_config.json`), то есть доступен лишь с самого телефона. Плагин переделывать не нужно — достаточно пробросить порт.

Доступ из интернета — вне рамок проекта (`development/mcp-kb/overview#secondary-goals`); всё ниже — для домашней сети или USB.

## Способ 1: SSH-туннель (рекомендуется)

На телефоне в Termux один раз:
```
pkg install openssh
passwd
```
и перед работой — `sshd` (слушает порт 8022). IP телефона — в настройках Wi-Fi или `ip addr` в Termux.

На ПК:
```
ssh -N -L 8888:127.0.0.1:8888 -p 8022 <ip-телефона>
```
Пока команда работает, `http://127.0.0.1:8888` на ПК ведёт на телефон.

- Трафик зашифрован, телефон не открывает Jupyter в сеть.
- Команды `claude mcp add …` из консоли Jupyter на телефоне и адреса в навыке `kb-index` подходят без изменений.
- На ПК порт 8888 должен быть свободен: если там запущен свой Jupyter — либо остановить его, либо взять другой локальный порт (`-L 8890:127.0.0.1:8888`) и поменять порт в адресе MCP-сервера.

## Способ 2: adb (USB или Wi-Fi)

На телефоне — включить отладку по USB (или беспроводную отладку) в настройках разработчика. На ПК (нужны platform-tools):
```
adb forward tcp:8888 tcp:8888
```
Эффект тот же, что у SSH-туннеля: `127.0.0.1:8888` на ПК ведёт на телефон, адреса не меняются. Удобно, если телефон и так подключён проводом; проброс пропадает при отключении — команду повторить.

## Способ 3: открыть Jupyter в домашнюю сеть

На телефоне в `jupyter_data/config/jupyter_server_config.json` заменить `"ip": "127.0.0.1"` на `"0.0.0.0"` и перезапустить. С ПК — `http://<ip-телефона>:8888/mcp/kb-admin?…&token=<токен>`.

- Проще всего, но токен и тексты идут по Wi-Fi открытым HTTP, порт виден всем в сети. Только для доверенной домашней сети.
- Плагин печатает адреса с `127.0.0.1` — в скопированных командах `claude mcp add` подставить IP телефона вручную.
- В навыке `kb-index` адрес правил `GET http://127.0.0.1:8888/mcp/docs/ru/admin` тоже указывает на ПК; он нужен только если `instructions` сервера не видны, но иметь в виду.

Если этот способ станет основным — сделать в плагине режим сетевого доступа, который печатает адреса с реальным IP (пока не сделано).

## Что учесть

- На телефоне должен работать Jupyter (`sh start_jupyter.sh`); `termux-wake-lock` в скрипте не даст ему уснуть с выключенным экраном, если стоит termux-api.
- Токен — с телефона (его `token.txt` или строка `[kb] token:`), не с ПК. Если на ПК уже есть серверы `kb` / `kb-admin` от своей базы — добавить телефонные под другими именами (например `kb-phone`) или сначала `claude mcp remove`.
- Правки индексатора пишутся в базу на телефоне, автокоммиты — там же; синхронизировать ничего не нужно.
- `start_jupyter.sh` в Termux живьём ещё не проверялся (на 2026-11-20): сначала убедиться, что он стартует и печатает строки `[kb]`.